# PrismLab — Modular Multi-Model Architecture

**Topology:** User request → intent router → six-model shared pool → **2-model document pipeline**, **3-model speech pipeline**, **3-model decision pipeline** → individual route comparisons → P6 integration → P7+ evaluation.

- **P1/P2 text and vision paths:** previously tested locally, but **pipeline winners not yet established**.
- **P3 Audio, P4 Omni, P5 d1:** planned integrations; multi-model routes are proposals to evaluate.
- The arrows show **logical routing and potential sequential execution**, *not all models loaded simultaneously on the device*.

Click a pipeline button to highlight its participating model(s); green dots in the pool indicate locally tested models. The comparison boxes intentionally show **Winner: not measured**.

In [ ]:
from pathlib import Path
from html import escape
from IPython.display import HTML, display

# Self-contained interactive SVG: no Mermaid CDN, no Manim, no external files required.
page = r'''<!doctype html><html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>PrismLab | Multi-model Architectures</title>
<style>
:root{color-scheme:dark;--bg:#07101e;--ink:#eaf2ff;--muted:#a5bad3;--a:#31e5b0;--b:#62b5ff;--c:#bb96ff}*{box-sizing:border-box}
html,body{width:100%;height:100%;margin:0;overflow:hidden;background:#07101e;color:var(--ink);font-family:Inter,system-ui,-apple-system,"Segoe UI",sans-serif}
.shell{height:100%;display:flex;flex-direction:column;overflow:hidden;background:radial-gradient(ellipse 58% 55% at 50% 41%,#11213b 0%,#07101e 100%);border:1px solid #253d5a}
header{flex:0 0 auto;min-height:60px;padding:10px 18px;display:flex;align-items:center;justify-content:space-between;gap:12px;border-bottom:1px solid #243b59}
h1{margin:0;font-size:17px;font-weight:760;white-space:nowrap}h1 span{color:#8eb4d8;font-weight:450}.sub{color:#9eafc8;font-size:11px;margin-top:4px}
.controls{display:flex;justify-content:flex-end;flex-wrap:wrap;gap:5px}button{font:inherit;font-size:11px;padding:7px 10px;border-radius:7px;color:#cbdcf4;cursor:pointer;border:1px solid #354f72;background:#13243b}button.active{border-color:#52a9f7;color:white;background:#20446b;box-shadow:0 0 10px #3daeff30}button:hover{border-color:#73c8ff}#pause{color:#a8f3de;border-color:#31695e}
.stage{min-height:0;flex:1;padding:0 9px;display:flex;align-items:center;justify-content:center;overflow:hidden}
#architecture{width:100%;height:100%;max-height:100%;max-width:100%;overflow:visible;display:block}
footer{flex:0 0 auto;min-height:40px;padding:7px 17px;border-top:1px solid #213956;display:flex;gap:12px;align-items:center;justify-content:space-between;color:#9fb5d3;font-size:10.5px}
footer .phases{display:flex;gap:5px;align-items:center;flex-wrap:wrap}footer .phases span{border:1px solid #2c4162;background:#10213a;padding:4px 7px;border-radius:5px}
.legend{white-space:nowrap}.legend b{color:#40eab7}.legend i{color:#b9c2d5;font-style:normal}
.wire{fill:none;stroke:#779cbd;stroke-width:2.1;opacity:.36;stroke-linecap:round;stroke-linejoin:round;vector-effect:non-scaling-stroke;transition:opacity .3s,stroke-width .3s}
.wire[data-track=core]{stroke:#84b9e8;opacity:.72}.wire[data-track=a]{stroke:var(--a)}.wire[data-track=b]{stroke:var(--b)}.wire[data-track=c]{stroke:var(--c)}
.wire.bus{stroke-dasharray:4 8;opacity:.5}
body[data-view=all] .wire:not(.bus),body[data-view=a] .wire[data-track=a],body[data-view=b] .wire[data-track=b],body[data-view=c] .wire[data-track=c]{opacity:.95;stroke-dasharray:7 10;stroke-width:2.7;animation:flow 1.5s linear infinite;filter:drop-shadow(0 0 4px currentColor)}
body[data-view=a] .wire[data-track=b],body[data-view=a] .wire[data-track=c],body[data-view=b] .wire[data-track=a],body[data-view=b] .wire[data-track=c],body[data-view=c] .wire[data-track=a],body[data-view=c] .wire[data-track=b]{opacity:.09}
@keyframes flow{to{stroke-dashoffset:-34}}body.paused .wire{animation-play-state:paused!important}
.node text{text-anchor:middle;fill:#eaf4ff;pointer-events:none;user-select:none}.node .frame{fill:#12243b;stroke:#54799c;stroke-width:1.6;vector-effect:non-scaling-stroke;transition:filter .25s,stroke .25s}
.node.common .frame{fill:#112540;stroke:#6c9dd4}.node.router .frame{fill:#172c50;stroke:#64b5ff;filter:drop-shadow(0 0 6px #3a9eff55)}
.node .title{font-size:18px;font-weight:730;letter-spacing:.3px}.node .hint{font-size:13.5px;fill:#b6cde1}.node .phase{font-size:12.5px;fill:#87a7c7}
.pool{fill:#0a1d31;stroke:#2c5574;stroke-width:1.4;stroke-dasharray:7 8}.pool-tab{fill:#0c233a;stroke:#326286}.pool-caption{fill:#9ac6e8;font-size:13px;letter-spacing:1px;font-weight:740}
.node.model{cursor:pointer}.model .model-title{font-size:18px;font-weight:770}.model .role{font-size:12.7px;fill:#abc3d9;letter-spacing:.7px}.model .model-dot{fill:#95aabf}.model.tested .model-dot{fill:#34e3b1;filter:drop-shadow(0 0 5px #30e6b2)}
.node.pipeline{cursor:pointer}.node.pipeline[data-track=a] .frame,.node.comparison[data-track=a] .frame{stroke:#22997f}.node.pipeline[data-track=b] .frame,.node.comparison[data-track=b] .frame{stroke:#4887bc}.node.pipeline[data-track=c] .frame,.node.comparison[data-track=c] .frame{stroke:#8b74bd}
.node .compare{font-size:14px;fill:#b7d2e8}.node .win{font-size:14.5px;font-weight:640;fill:#f1f6ff}
.node.integration .frame,.node.evaluation .frame{fill:#122743;stroke:#569ce7;filter:drop-shadow(0 0 6px #3d91ff3d)}
body[data-view=all] .node.model.tested .frame,body[data-view=all] .node.pipeline[data-track=a] .frame,
body[data-view=a] .node.pipeline[data-track=a] .frame,body[data-view=a] .node.comparison[data-track=a] .frame,
body[data-view=b] .node.pipeline[data-track=b] .frame,body[data-view=b] .node.comparison[data-track=b] .frame,
body[data-view=c] .node.pipeline[data-track=c] .frame,body[data-view=c] .node.comparison[data-track=c] .frame{stroke-width:2.2;filter:drop-shadow(0 0 8px #4a9cf976)}
body[data-view=a] .node.model[data-members~="a"] .frame,body[data-view=b] .node.model[data-members~="b"] .frame,body[data-view=c] .node.model[data-members~="c"] .frame{stroke:#6be9c5;stroke-width:2.2;filter:drop-shadow(0 0 8px #65b8ffc0)}
body[data-view=a] .node.pipeline:not([data-track=a]),body[data-view=a] .node.comparison:not([data-track=a]),body[data-view=b] .node.pipeline:not([data-track=b]),body[data-view=b] .node.comparison:not([data-track=b]),body[data-view=c] .node.pipeline:not([data-track=c]),body[data-view=c] .node.comparison:not([data-track=c]){opacity:.32}
body[data-view=a] .node.model:not([data-members~="a"]),body[data-view=b] .node.model:not([data-members~="b"]),body[data-view=c] .node.model:not([data-members~="c"]){opacity:.3}
.node.model:hover .frame,.node.pipeline:hover .frame,.node:focus .frame{stroke:#f2faff!important;filter:drop-shadow(0 0 9px #8cd9ff96)!important}
.particle{filter:drop-shadow(0 0 4px currentColor)}@media(max-width:850px){header{padding:8px 10px;min-height:62px}h1{font-size:15px}button{font-size:10px;padding:5px 7px}footer{font-size:9px;padding:6px 8px}.phases span{padding:3px 5px}}
@media(max-width:640px){header{flex-direction:column;align-items:flex-start}footer{flex-wrap:wrap}.controls{justify-content:flex-start}}
@media(prefers-reduced-motion:reduce){.wire{animation:none!important}}
</style></head><body data-view="all"><div class="shell">
<header><div><h1>PrismLab <span>/ Modular Multi-Model Pipeline</span></h1><div class="sub">Shared six-model pool → 2-model and 3-model routes → compare → integrate</div></div>
<div class="controls"><button data-select="all" class="active">All routes</button><button data-select="a">Document · 2</button><button data-select="b">Speech · 3</button><button data-select="c">Decision · 3</button><button id="pause">Pause</button></div></header>
<div class="stage"><svg id="architecture" viewBox="0 0 1440 775" role="img" aria-label="Six models feed three configurable pipelines of two or three models each, then per-pipeline comparisons, unified workbench and evaluation.">
<defs><filter id="glow"><feGaussianBlur stdDeviation="3" result="b"/><feMerge><feMergeNode in="b"/><feMergeNode in="SourceGraphic"/></feMerge></filter></defs>
<g id="wires"><path class="wire " data-track="core" d="M720 68 V98"/><path class="wire " data-track="core" d="M720 149 V180"/><path class="wire bus" data-track="core" d="M147 188 H1292"/><path class="wire " data-track="core" d="M720 180 V188"/><path class="wire " data-track="core" d="M147 188 V236"/><path class="wire " data-track="core" d="M376 188 V236"/><path class="wire " data-track="core" d="M605 188 V236"/><path class="wire " data-track="core" d="M834 188 V236"/><path class="wire " data-track="core" d="M1063 188 V236"/><path class="wire " data-track="core" d="M1292 188 V236"/><path class="wire " data-track="a" d="M147 312 V335 Q147 369 249.0 375 L249.0 390"/><path class="wire " data-track="a" d="M376 312 V335 Q376 369 271.0 375 L271.0 390"/><path class="wire " data-track="a" d="M260 472 V506"/><path class="wire " data-track="a" d="M260 565 V588 Q260 605 720 613"/><path class="wire " data-track="b" d="M147 312 V335 Q147 369 698.0 375 L698.0 390"/><path class="wire " data-track="b" d="M605 312 V335 Q605 369 720.0 375 L720.0 390"/><path class="wire " data-track="b" d="M834 312 V335 Q834 369 742.0 375 L742.0 390"/><path class="wire " data-track="b" d="M720 472 V506"/><path class="wire " data-track="b" d="M720 565 V588 Q720 605 720 613"/><path class="wire " data-track="c" d="M834 312 V335 Q834 369 1158.0 375 L1158.0 390"/><path class="wire " data-track="c" d="M1063 312 V335 Q1063 369 1180.0 375 L1180.0 390"/><path class="wire " data-track="c" d="M1292 312 V335 Q1292 369 1202.0 375 L1202.0 390"/><path class="wire " data-track="c" d="M1180 472 V506"/><path class="wire " data-track="c" d="M1180 565 V588 Q1180 605 720 613"/><path class="wire " data-track="core" d="M720 613 V629"/><path class="wire " data-track="core" d="M720 681 V703"/></g><g id="nodes"><g class="node common"><rect class="frame" x="583" y="20" width="274" height="48" rx="12"/><text class="title" x="720" y="42">USER REQUEST</text><text class="hint" x="720" y="58">Text · Image · Audio · Video</text></g><g class="node common router"><rect class="frame" x="581" y="98" width="278" height="51" rx="13"/><text class="title" x="720" y="120">INTENT ROUTER</text><text class="hint" x="720" y="139">Select one or multiple models</text></g><rect class="pool" x="27" y="202" width="1386" height="136" rx="17"/><rect class="pool-tab" x="52" y="191" width="270" height="25" rx="7"/><text class="pool-caption" x="66" y="209">SIX-MODEL SHARED POOL</text><g class="node model tested" data-members="a b" data-index="0" tabindex="0" role="button" aria-label="Model LFM3-1.7B">
<rect class="frame" x="50" y="236" width="194" height="76" rx="13"/>
<circle cx="66" cy="251" r="3.5" class="model-dot"/>
<text class="model-title" x="147" y="266">LFM3-1.7B</text>
<text class="role" x="147" y="289">TEXT</text></g><g class="node model tested" data-members="a" data-index="1" tabindex="0" role="button" aria-label="Model LFM3-VL-2B">
<rect class="frame" x="279" y="236" width="194" height="76" rx="13"/>
<circle cx="295" cy="251" r="3.5" class="model-dot"/>
<text class="model-title" x="376" y="266">LFM3-VL-2B</text>
<text class="role" x="376" y="289">VISION</text></g><g class="node model planned" data-members="b" data-index="2" tabindex="0" role="button" aria-label="Model LFM3-Audio-2B">
<rect class="frame" x="508" y="236" width="194" height="76" rx="13"/>
<circle cx="524" cy="251" r="3.5" class="model-dot"/>
<text class="model-title" x="605" y="266">LFM3-Audio-2B</text>
<text class="role" x="605" y="289">AUDIO</text></g><g class="node model planned" data-members="b c" data-index="3" tabindex="0" role="button" aria-label="Model LFM3-Omni-2B">
<rect class="frame" x="737" y="236" width="194" height="76" rx="13"/>
<circle cx="753" cy="251" r="3.5" class="model-dot"/>
<text class="model-title" x="834" y="266">LFM3-Omni-2B</text>
<text class="role" x="834" y="289">OMNI</text></g><g class="node model planned" data-members="c" data-index="4" tabindex="0" role="button" aria-label="Model d1-2B">
<rect class="frame" x="966" y="236" width="194" height="76" rx="13"/>
<circle cx="982" cy="251" r="3.5" class="model-dot"/>
<text class="model-title" x="1063" y="266">d1-2B</text>
<text class="role" x="1063" y="289">DECISION</text></g><g class="node model planned" data-members="c" data-index="5" tabindex="0" role="button" aria-label="Model d1-Omni-2B">
<rect class="frame" x="1195" y="236" width="194" height="76" rx="13"/>
<circle cx="1211" cy="251" r="3.5" class="model-dot"/>
<text class="model-title" x="1292" y="266">d1-Omni-2B</text>
<text class="role" x="1292" y="289">A/V DECISION</text></g><g class="node pipeline" data-track="a" tabindex="0" role="button" aria-label="Select DOCUMENT → ANSWER workflow">
<rect class="frame" x="57" y="390" width="406" height="82" rx="14"/>
<text class="title" x="260" y="414">DOCUMENT → ANSWER</text><text class="hint" x="260" y="439">Text + Vision</text><text class="phase" x="260" y="459">2 models · P1–P2 tested</text></g><g class="node comparison" data-track="a"><rect class="frame" x="57" y="506" width="406" height="59" rx="13"/>
<text class="compare" x="260" y="527">VL-only vs VL → Text</text><text class="win" x="260" y="550">Winner: not measured</text></g><g class="node pipeline" data-track="b" tabindex="0" role="button" aria-label="Select SPEECH → RESPONSE workflow">
<rect class="frame" x="517" y="390" width="406" height="82" rx="14"/>
<text class="title" x="720" y="414">SPEECH → RESPONSE</text><text class="hint" x="720" y="439">Audio + Text + Omni</text><text class="phase" x="720" y="459">3 models · P3–P4 planned</text></g><g class="node comparison" data-track="b"><rect class="frame" x="517" y="506" width="406" height="59" rx="13"/>
<text class="compare" x="720" y="527">Audio vs Omni vs 3-model chain</text><text class="win" x="720" y="550">Winner: not measured</text></g><g class="node pipeline" data-track="c" tabindex="0" role="button" aria-label="Select OBSERVE → DECIDE workflow">
<rect class="frame" x="977" y="390" width="406" height="82" rx="14"/>
<text class="title" x="1180" y="414">OBSERVE → DECIDE</text><text class="hint" x="1180" y="439">Omni + d1 + d1-Omni</text><text class="phase" x="1180" y="459">3 models · P5 planned</text></g><g class="node comparison" data-track="c"><rect class="frame" x="977" y="506" width="406" height="59" rx="13"/>
<text class="compare" x="1180" y="527">d1 vs d1-Omni vs chained route</text><text class="win" x="1180" y="550">Winner: not measured</text></g><g class="node common integration"><rect class="frame" x="366" y="629" width="708" height="52" rx="14"/>
<text class="title" x="720" y="651">P6 · UNIFIED WORKBENCH</text><text class="hint" x="720" y="669">Orchestrate specialist paths and compare against single-model baselines</text></g><g class="node common evaluation"><rect class="frame" x="366" y="703" width="708" height="51" rx="14"/>
<text class="title" x="720" y="726">P7+ · VALIDATED EVALUATION</text><text class="hint" x="720" y="743">Accuracy · Latency / TTFT · Memory · Robustness · Calibrated abstention</text></g></g><g id="particles" pointer-events="none"></g></svg></div>
<footer><div class="phases"><span>P1 Router + Text</span><span>P2 Vision</span><span>P3 Audio</span><span>P4 Omni</span><span>P5 d1 Decisions</span><span>P6 Integration</span><span>P7+ Evaluation</span></div><span class="legend"><b>●</b> tested model · <i>●</i> planned · winners TBD</span></footer>
</div>
<script>(()=>{
const body=document.body, svg=document.getElementById('architecture'),layer=document.getElementById('particles');
const buttons=[...document.querySelectorAll('[data-select]')];let selected='all',paused=false,last=0,elapsed=0;
const colors={core:'#8fc8ff',a:'#44e8b7',b:'#74bbff',c:'#c5a4ff'};
const motions=[...svg.querySelectorAll('.wire:not(.bus)')].map((p,i)=>{
 const track=p.dataset.track,dot=document.createElementNS('http://www.w3.org/2000/svg','circle');
 dot.setAttribute('r',track==='core'?'2.9':'3.6');dot.setAttribute('fill',colors[track]);dot.setAttribute('class','particle');layer.appendChild(dot);
 return {p,track,dot,len:p.getTotalLength(),offset:(i*.61803398875)%1,duration:1900+(i*167)%850};
});
function switchTo(value){selected=value;body.dataset.view=value;buttons.forEach(b=>{const on=b.dataset.select===value;b.classList.toggle('active',on);b.setAttribute('aria-pressed',on?'true':'false')});}
buttons.forEach(b=>b.addEventListener('click',()=>switchTo(b.dataset.select)));
svg.querySelectorAll('.pipeline').forEach(n=>{n.addEventListener('click',()=>switchTo(n.dataset.track));n.addEventListener('keydown',e=>{if(e.key==='Enter'||e.key===' '){e.preventDefault();switchTo(n.dataset.track)}})});
svg.querySelectorAll('.model').forEach(n=>{const preferred=n.dataset.members.split(' ')[0];n.addEventListener('click',()=>switchTo(preferred));n.addEventListener('keydown',e=>{if(e.key==='Enter'||e.key===' '){e.preventDefault();switchTo(preferred)}})});
document.querySelector('#pause').addEventListener('click',e=>{paused=!paused;body.classList.toggle('paused',paused);e.currentTarget.textContent=paused?'Resume':'Pause'});
function animate(t){if(!last)last=t;const dt=Math.min(40,t-last);last=t;if(!paused&&!matchMedia('(prefers-reduced-motion: reduce)').matches)elapsed+=dt;
for(const m of motions){const shown=m.track==='core'||selected==='all'||selected===m.track;m.dot.style.opacity=shown?'1':'0';if(!shown)continue;
const pt=m.p.getPointAtLength((((elapsed/m.duration)+m.offset)%1)*m.len);m.dot.setAttribute('cx',pt.x);m.dot.setAttribute('cy',pt.y);}
requestAnimationFrame(animate)}requestAnimationFrame(animate);
})();</script></body></html>'''
Path("PrismLab_MultiModel_Architecture.html").write_text(page, encoding="utf-8")
frame = ('<iframe sandbox="allow-scripts" '
         'style="width:100%;height:790px;border:0;border-radius:12px;background:#07101e" '
         'srcdoc="' + escape(page, quote=True) + '"></iframe>')
display(HTML(frame))
print("Saved PrismLab_MultiModel_Architecture.html")
